In [1]:
# Mount Google Drive (if needed for saving model and data)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import re
import string
import tensorflow_datasets as tfds
from tensorflow.keras.layers import Embedding, LSTM, Dense, GRU
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
import os


2025-02-19 02:51:30.657982: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-02-19 02:51:30.664711: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-02-19 02:51:30.681346: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1739911890.705286   30275 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1739911890.714053   30275 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-19 02:51:30.739362: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU ins

In [2]:
# Load dataset
def load_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    data = []
    for line in lines:
        parts = line.strip().split('\t')
        if len(parts) >= 2:  # Ensuring valid data
            data.append((parts[0], parts[1]))
    return pd.DataFrame(data, columns=['English', 'Bangla'])# Load dataset
def load_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    data = []
    for line in lines:
        parts = line.strip().split('\t')
        if len(parts) >= 2:  # Ensuring valid data
            data.append((parts[0], parts[1]))
    return pd.DataFrame(data, columns=['English', 'Bangla'])
data_path = '/media/hmb/hdd2/Ongoing_Projects/NLP/Play_with_NLP/Project_1/Bangla_English_Translator/machine_translation/data/dataset.txt'  # Update with actual file path
df = load_dataset(data_path)
print("Dataset Sample:")
print(df.head())

Dataset Sample:
  English  Bangla
0     Go.    যাও।
1     Go.    যান।
2     Go.     যা।
3    Run!  পালাও!
4    Run!  পালান!


In [3]:
# Preprocessing functions
def clean_text(text):
    text = text.lower()
    text = re.sub(f"[{string.punctuation}]", "", text)
    return text

df['English'] = df['English'].apply(clean_text)
df['Bangla'] = df['Bangla'].apply(clean_text)

In [4]:
print(df['English'].head(10))

0      go
1      go
2      go
3     run
4     run
5     who
6     wow
7    fire
8    help
9    help
Name: English, dtype: object


In [5]:
print(df['Bangla'].head(10))

0      যাও।
1      যান।
2       যা।
3     পালাও
4     পালান
5        কে
6       বাহ
7      আগুন
8    বাঁচাও
9    বাঁচান
Name: Bangla, dtype: object


In [6]:
len(df['English'])

6509

In [7]:
len(df['Bangla'])

6509

In [8]:
# Tokenization
eng_tokenizer = tfds.deprecated.text.SubwordTextEncoder.build_from_corpus(
    df['English'], target_vocab_size=2**13)
bn_tokenizer = tfds.deprecated.text.SubwordTextEncoder.build_from_corpus(
    df['Bangla'], target_vocab_size=2**13)


In [9]:
print(eng_tokenizer)

<SubwordTextEncoder vocab_size=3742>


In [10]:
print(bn_tokenizer)

<SubwordTextEncoder vocab_size=1464>


In [11]:
# Convert text to sequences
def encode_texts(eng, bn):
    eng_seq = eng_tokenizer.encode(eng)
    bn_seq = bn_tokenizer.encode(bn)
    return eng_seq, bn_seq

df['eng_seq'], df['bn_seq'] = zip(*df.apply(lambda row: encode_texts(row['English'], row['Bangla']), axis=1))

In [12]:
df['eng_seq'].head(10)

0     [106]
1     [106]
2     [106]
3     [506]
4     [506]
5    [1913]
6    [1885]
7     [475]
8     [310]
9     [310]
Name: eng_seq, dtype: object

In [13]:
df['bn_seq'].head(10)

0        [13, 1, 88, 15]
1         [13, 1, 7, 15]
2               [13, 40]
3     [24, 1, 18, 1, 88]
4      [24, 1, 18, 1, 7]
5                 [8, 2]
6            [19, 1, 52]
7           [105, 16, 7]
8    [19, 70, 25, 1, 88]
9     [19, 70, 25, 1, 7]
Name: bn_seq, dtype: object

In [14]:
# Padding sequences
max_len = max(max(df['eng_seq'].apply(len)), max(df['bn_seq'].apply(len)))
df['eng_seq'] = df['eng_seq'].apply(lambda x: x + [0] * (max_len - len(x)))
df['bn_seq'] = df['bn_seq'].apply(lambda x: x + [0] * (max_len - len(x)))


In [16]:
# Splitting dataset
X_train, X_test, y_train, y_test = train_test_split(df['eng_seq'].tolist(), df['bn_seq'].tolist(), test_size=0.1)
X_train, X_test, y_train, y_test = map(np.array, [X_train, X_test, y_train, y_test])


In [17]:

# Build Seq2Seq Model
class Seq2Seq(Model):
    def __init__(self, vocab_size, embedding_dim, units):
        super(Seq2Seq, self).__init__()
        self.encoder = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.gru = tf.keras.layers.GRU(units, return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.Dense(vocab_size, activation='softmax')

    def call(self, inputs):
        x = self.encoder(inputs)
        x, state = self.gru(x)
        x = self.decoder(x)
        return x


In [18]:

# Define hyperparameters
embedding_dim = 256
units = 512
vocab_size = max(eng_tokenizer.vocab_size, bn_tokenizer.vocab_size) + 1


In [19]:
# Initialize and compile model
model = Seq2Seq(vocab_size, embedding_dim, units)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])


2025-02-19 02:52:16.486822: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [20]:
# Train model
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=32)


Epoch 1/10


2025-02-19 02:52:22.847668: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 245301248 exceeds 10% of free system memory.


  1/184 ━━━━━━━━━━━━━━━━━━━━ 14:17 5s/step - accuracy: 0.0000e+00 - loss: 8.2338

2025-02-19 02:52:24.261204: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 245301248 exceeds 10% of free system memory.


  2/184 ━━━━━━━━━━━━━━━━━━━━ 3:30 1s/step - accuracy: 0.2009 - loss: 8.1833     

...

  3/184 ━━━━━━━━━━━━━━━━━━━━ 3:30 1s/step - accuracy: 0.3120 - loss: 8.1116

2025-02-19 02:52:26.665847: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 245301248 exceeds 10% of free system memory.


  4/184 ━━━━━━━━━━━━━━━━━━━━ 3:37 1s/step - accuracy: 0.3830 - loss: 8.0022

2025-02-19 02:52:27.921387: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 245301248 exceeds 10% of free system memory.


184/184 ━━━━━━━━━━━━━━━━━━━━ 246s 1s/step - accuracy: 0.7760 - loss: 1.9852 - val_accuracy: 0.8132 - val_loss: 0.9986
Epoch 2/10
184/184 ━━━━━━━━━━━━━━━━━━━━ 165s 897ms/step - accuracy: 0.8153 - loss: 0.9510 - val_accuracy: 0.8162 - val_loss: 0.9370
Epoch 3/10
184/184 ━━━━━━━━━━━━━━━━━━━━ 230s 1s/step - accuracy: 0.8192 - loss: 0.9009 - val_accuracy: 0.8177 - val_loss: 0.9294
Epoch 4/10
184/184 ━━━━━━━━━━━━━━━━━━━━ 231s 1s/step - accuracy: 0.8212 - loss: 0.8746 - val_accuracy: 0.8184 - val_loss: 0.9128
Epoch 5/10
184/184 ━━━━━━━━━━━━━━━━━━━━ 306s 2s/step - accuracy: 0.8238 - loss: 0.8545 - val_accuracy: 0.8188 - val_loss: 0.9054
Epoch 6/10
184/184 ━━━━━━━━━━━━━━━━━━━━ 300s 2s/step - accuracy: 0.8219 - loss: 0.8541 - val_accuracy: 0.8201 - val_loss: 0.8986
Epoch 7/10
184/184 ━━━━━━━━━━━━━━━━━━━━ 237s 1s/step - accuracy: 0.8262 - loss: 0.8230 - val_accuracy: 0.8207 - val_loss: 0.8961
Epoch 8/10
184/184 ━━━━━━━━━━━━━━━━━━━━ 249s 1s/step - accuracy: 0.8262 - loss: 0.8179 - val_accuracy: 0.

In [22]:

# Save model
model.save('/media/hmb/hdd2/Ongoing_Projects/NLP/Play_with_NLP/Project_1/Bangla_English_Translator/machine_translation/models.h5')

In [25]:
# Function to translate English to Bangla
def translate(sentence):
    sentence = clean_text(sentence)
    seq = eng_tokenizer.encode(sentence)
    seq = seq + [0] * (max_len - len(seq))
    seq = np.array([seq])
    pred = model.predict(seq)
    pred_seq = np.argmax(pred, axis=-1)[0]
    translated_text = bn_tokenizer.decode([i for i in pred_seq if i != 0])
    return translated_text

# Example translation
example = "How are you"
print(f"English: {example}")
print(f"Bangla Translation: {translate(example)}")


English: How are you
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
Bangla Translation: আপনাকাাা
